# F7 : Détection d'anomalies sur le S&P 500 — Graphiques et interprétations

**WAHHABI Zakaria** · 23009144 · G3 · Finance

Ce notebook regroupe les huit graphiques du projet, chacun suivi de son interprétation.


In [1]:
# Préparation des calculs nécessaires aux graphiques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

GRAINE = 42
np.random.seed(GRAINE)

from google.colab import drive
drive.mount("/content/drive")
FICHIER = Path("/content/drive/MyDrive/SP500 (1).csv")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

# Chargement et nettoyage
df = pd.read_csv(FICHIER)
df.columns = ["date", "cloture"]
df["date"] = pd.to_datetime(df["date"])
df["cloture"] = pd.to_numeric(df["cloture"], errors="coerce")
df = df.dropna(subset=["cloture"]).set_index("date").sort_index()

# Variables : rendement et volatilité des 20 jours précédents
df["rendement"] = np.log(df["cloture"]).diff()
df["volatilite_20j"] = df["rendement"].rolling(20).std().shift(1)
donnees = df.dropna(subset=["rendement", "volatilite_20j"]).copy()
VARIABLES = ["rendement", "volatilite_20j"]

# Découpage chronologique
DEBUT_VALID, DEBUT_TEST = "2020-01-01", "2022-01-01"
train = donnees[donnees.index < DEBUT_TEST]
test = donnees[donnees.index >= DEBUT_TEST].copy()
train_int = train[train.index < DEBUT_VALID]
valid = train[train.index >= DEBUT_VALID]

def creer_modele(taux, graine=GRAINE):
    return Pipeline([("standardisation", StandardScaler()),
                     ("iforest", IsolationForest(n_estimators=300, contamination=taux, random_state=graine))])

EVENEMENTS = {
    "Volmageddon": ("2018-02-01", "2018-02-12"),
    "Correction d'octobre 2018": ("2018-10-01", "2018-10-31"),
    "Chute de décembre 2018": ("2018-12-01", "2018-12-31"),
    "Guerre commerciale, août 2019": ("2019-08-01", "2019-08-23"),
    "Krach du Covid-19": ("2020-02-20", "2020-04-30"),
    "Rechute de juin 2020": ("2020-06-08", "2020-06-15"),
    "Invasion de l'Ukraine": ("2022-02-24", "2022-03-09"),
    "Inflation et Fed, mai-juin 2022": ("2022-05-01", "2022-06-17"),
    "Inflation (CPI), 13 sept. 2022": ("2022-09-13", "2022-09-16"),
    "Inflation (CPI), 10 nov. 2022": ("2022-11-10", "2022-11-11"),
    "Faillite de SVB": ("2023-03-08", "2023-03-17"),
    "Débouclage du carry trade sur le yen": ("2024-08-01", "2024-08-09"),
    "Droits de douane américains": ("2025-04-02", "2025-04-15"),
}

def etiqueter(index):
    etiquettes = pd.Series("Non identifié", index=index)
    for nom, (debut, fin) in EVENEMENTS.items():
        etiquettes[(index >= debut) & (index <= fin)] = nom
    return etiquettes

def evaluer(detecte, index):
    detecte = np.asarray(detecte)
    etiquettes = etiqueter(index)
    dans_evenement = (etiquettes != "Non identifié").values
    n = int(detecte.sum())
    precision = dans_evenement[detecte].mean() if n > 0 else np.nan
    hasard = dans_evenement.mean()
    evenements_periode = set(etiquettes[dans_evenement])
    trouves = set(etiquettes[detecte & dans_evenement])
    rappel = len(trouves) / len(evenements_periode) if evenements_periode else np.nan
    return {
        "Jours détectés": n,
        "Proportion (%)": round(100 * n / len(index), 2),
        "Précision événementielle (%)": round(100 * precision, 1),
        "Niveau du hasard (%)": round(100 * hasard, 1),
        "Ratio / hasard": round(precision / hasard, 2) if hasard > 0 else np.nan,
        "Rappel événementiel (%)": round(100 * rappel, 1),
    }

def regle_3sigma(apprentissage, a_noter):
    mu, sigma = apprentissage["rendement"].mean(), apprentissage["rendement"].std()
    return ((a_noter["rendement"] - mu).abs() > 3 * sigma).values

# Réglage du taux sur la validation
TAUX_CANDIDATS = [0.005, 0.01, 0.02, 0.05]
lignes = []
for taux in TAUX_CANDIDATS:
    m = creer_modele(taux).fit(train_int[VARIABLES])
    for nom, b in [("Apprentissage", train_int), ("Validation", valid)]:
        lignes.append({"Taux (%)": 100 * taux, "Période": nom,
                       **evaluer(m.predict(b[VARIABLES]) == -1, b.index)})
reglage = pd.DataFrame(lignes)
choix = (reglage[reglage["Période"] == "Validation"]
         .sort_values(["Précision événementielle (%)", "Taux (%)"], ascending=[False, True]).iloc[0])
TAUX = choix["Taux (%)"] / 100

# Modèle final et application au test
modele = creer_modele(TAUX).fit(train[VARIABLES])
detecte_train = modele.predict(train[VARIABLES]) == -1
test["score"] = -modele.decision_function(test[VARIABLES])
test["anomalie"] = modele.predict(test[VARIABLES]) == -1
test["anomalie_3sigma"] = regle_3sigma(train, test)
print("Données prêtes :", len(donnees), "jours | taux retenu :", f"{TAUX:.1%}")

Mounted at /content/drive
Données prêtes : 2492 jours | taux retenu : 0.5%


## Graphique 1 : Clôture quotidienne du S&P 500

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df["cloture"], color="steelblue", lw=1)
ax.set_title("Clôture quotidienne du S&P 500")
ax.set_xlabel("Date")
ax.set_ylabel("Indice (points)")
plt.tight_layout()
plt.savefig(FIGURES / "fig1_cours.png", dpi=150)
plt.show()

**Interprétation.** L'indice progresse d'environ 2 130 points fin 2016 à un maximum de 7 818,9 points, avec des chutes brutales, la plus forte en février-mars 2020 (Covid). Le niveau du prix suit une tendance : une valeur élevée traduit la croissance, pas une anomalie. Il faut donc travailler sur les **variations** (rendements), pas sur le prix.

## Graphique 2 : Rendements quotidiens dans le temps

In [ ]:
rendement_pct = 100 * np.log(df["cloture"]).diff()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rendement_pct.index, rendement_pct, color="grey", lw=0.6)
ax.set_title("Rendements logarithmiques quotidiens du S&P 500")
ax.set_xlabel("Date")
ax.set_ylabel("Rendement (%)")
plt.tight_layout()
plt.savefig(FIGURES / "fig2_rendements.png", dpi=150)
plt.show()

**Interprétation.** Les rendements oscillent la plupart du temps autour de zéro, mais les fortes variations arrivent **en grappes** : mars-avril 2020, l'année 2022 et avril 2025. C'est le regroupement de la volatilité. Une même variation n'a donc pas le même sens selon que le marché est calme ou agité, d'où l'ajout de la volatilité des 20 jours précédents comme deuxième variable.

## Graphique 3 : Distribution des rendements comparée à une loi normale

In [ ]:
rendement_pct = 100 * np.log(df["cloture"]).diff()
r = rendement_pct.dropna()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r, bins=120, density=True, color="steelblue", alpha=0.7, label="Rendements observés")
x = np.linspace(-6, 6, 400)
ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), "r--", label="Loi normale de même moyenne et écart-type")
ax.set_xlim(-6, 6)
ax.set_title("Distribution des rendements quotidiens comparée à une loi normale")
ax.set_xlabel("Rendement quotidien (%)")
ax.set_ylabel("Densité")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig3_distribution.png", dpi=150)
plt.show()
z = (r - r.mean()) / r.std()
print(f"Kurtosis en excès : {stats.kurtosis(r):.1f} (0 pour une loi normale)")
print(f"Jours à plus de 3 écarts-types : {(z.abs() > 3).sum()} sur {len(r)} "
      f"({100 * (z.abs() > 3).mean():.2f} %), contre environ 0,27 % attendu pour une loi normale")

**Interprétation.** Le kurtosis en excès vaut **16,6** (0 pour une loi normale). **37 jours sur 2 512 (1,47 %)** dépassent 3 écarts-types, contre 0,27 % attendus : environ **5 fois plus** que prévu. Ce sont les **queues épaisses** : les journées extrêmes sont réelles et se répètent, ce qui justifie de chercher à les détecter.

## Graphique 4 : Nuage rendement / volatilité (apprentissage)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train["rendement"] * 100, train["volatilite_20j"] * 100, s=8, alpha=0.5, color="steelblue")
ax.set_title("Apprentissage : chaque point est une journée de bourse")
ax.set_xlabel("Rendement du jour (%)")
ax.set_ylabel("Volatilité des 20 jours précédents (%)")
plt.tight_layout()
plt.savefig(FIGURES / "fig4_nuage.png", dpi=150)
plt.show()
print(f"Volatilité 20j maximale : apprentissage {100 * train['volatilite_20j'].max():.2f} % | test {100 * test['volatilite_20j'].max():.2f} %")

**Interprétation.** Chaque point est une journée de bourse. La grande majorité forme un nuage dense : rendement proche de zéro et volatilité faible. Les points isolés sur les bords sont les candidats anomalies. Les points les plus éloignés viennent du krach du Covid : la volatilité y atteint **6,17 %**, contre **3,18 %** au maximum sur la période de test. Cet écart explique la suite des résultats.

## Graphique 5 : Courbe de validation (réglage du taux d'anomalies)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for nom, couleur in [("Apprentissage", "tab:blue"), ("Validation", "tab:orange")]:
    sous = reglage[reglage["Période"] == nom]
    ax.plot(sous["Taux (%)"], sous["Précision événementielle (%)"], "o-", color=couleur, label=nom)
    ax.axhline(sous["Niveau du hasard (%)"].iloc[0], color=couleur, ls=":", label=f"Hasard ({nom.lower()})")
ax.set_xlabel("Taux d'anomalies, paramètre contamination (%)")
ax.set_ylabel("Précision événementielle (%)")
ax.set_title("Courbe de validation de l'Isolation Forest")
ax.set_ylim(0, 105)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig5_courbe_validation.png", dpi=150)
plt.show()
display(reglage)

**Interprétation.** Sur 4 configurations, le taux retenu est **0,5 %**, avec une précision événementielle de **96,8 %** en validation contre **11,1 %** pour le hasard (ratio 8,7). La précision baisse quand le taux augmente, sur les deux courbes : un taux plus élevé ajoute des journées ordinaires. Elle est plus forte en validation (96,8 %) qu'en apprentissage (75 %) parce que la validation 2020-2021 contient le Covid, très facile à détecter : le modèle y signale d'ailleurs 6,1 % des jours. Ce réglage est donc dominé par un seul épisode.

## Graphique 6 : Frise chronologique des anomalies

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, col, unite in [(axes[0], "cloture", "Indice (points)"), (axes[1], "rendement", "Rendement (%)")]:
    facteur = 100 if col == "rendement" else 1
    ax.plot(donnees.index, donnees[col] * facteur, color="grey", lw=0.7)
    ax.scatter(train.index[detecte_train], train.loc[detecte_train, col] * facteur, color="orange", s=25, zorder=3, label="Anomalie (apprentissage)")
    ax.scatter(test.index[test["anomalie"]], test.loc[test["anomalie"], col] * facteur, color="red", s=30, zorder=4, label="Anomalie (test)")
    ax.scatter(test.index[test["anomalie_3sigma"]], test.loc[test["anomalie_3sigma"], col] * facteur, marker="x", color="blue", s=40, zorder=5, label="Règle des 3 écarts-types (test)")
    for i, (debut, fin) in enumerate(EVENEMENTS.values()):
        ax.axvspan(pd.Timestamp(debut), pd.Timestamp(fin), color="gold", alpha=0.3, label="Événement documenté" if i == 0 else None)
    ax.axvline(pd.Timestamp(DEBUT_TEST), color="black", ls="--", label="Début du test" if ax is axes[0] else None)
    ax.set_ylabel(unite)
axes[0].set_title("S&P 500 : journées atypiques détectées par l'Isolation Forest")
axes[1].set_xlabel("Date")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig6_frise.png", dpi=150)
plt.show()
print("Anomalies de l'apprentissage :", [d.strftime("%Y-%m-%d") for d in train.index[detecte_train]])
print("Anomalies Isolation Forest au test :", int(test["anomalie"].sum()), "| règle des 3 écarts-types :", int(test["anomalie_3sigma"].sum()))

**Interprétation.** Les **7 anomalies** de l'apprentissage tombent toutes en mars-avril 2020 (krach du Covid). Sur le test (2022-2026), l'Isolation Forest ne détecte **aucune** journée, alors que la règle des 3 écarts-types en signale **10** : en 2022 (inflation, Fed), en avril 2025 (droits de douane américains) et le 5 novembre 2026. Plusieurs bandes jaunes du test ne contiennent donc aucun point rouge : le modèle manque les 7 événements de la période.

## Graphique 7 : Carte du score d'anomalie appliquée au test

In [ ]:
xx, yy = np.meshgrid(np.linspace(donnees["rendement"].min() * 1.1, donnees["rendement"].max() * 1.1, 250),
                     np.linspace(0, donnees["volatilite_20j"].max() * 1.1, 250))
grille = pd.DataFrame({"rendement": xx.ravel(), "volatilite_20j": yy.ravel()})
zz = -modele.decision_function(grille).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(8, 5.5))
fond = ax.contourf(xx * 100, yy * 100, zz, levels=20, cmap="Reds", alpha=0.45)
plt.colorbar(fond, ax=ax, label="Score d'anomalie (plus élevé = plus atypique)")
normal = ~test["anomalie"]
ax.scatter(test.loc[normal, "rendement"] * 100, test.loc[normal, "volatilite_20j"] * 100, s=8, alpha=0.5, color="steelblue", label="Test : normal")
ax.scatter(test.loc[~normal, "rendement"] * 100, test.loc[~normal, "volatilite_20j"] * 100, s=40, color="red", edgecolor="k", label="Test : anomalie")
ax.set_xlabel("Rendement du jour (%)")
ax.set_ylabel("Volatilité des 20 jours précédents (%)")
ax.set_title("Score d'anomalie appris sur 2016-2021, appliqué au test")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig7_score.png", dpi=150)
plt.show()
print(f"Score maximal au test : {test['score'].max():.3f} (seuil de décision : 0)")

**Interprétation.** Les zones foncées sont celles que le modèle juge atypiques : très forts rendements, surtout avec une volatilité précédente faible. Toutes les journées du test restent sous le seuil : le score maximal vaut **−0,049** (seuil de décision : 0), atteint le 9 avril 2025 (+9,09 %). Le modèle a placé sa frontière autour des journées du Covid, si bien que les chocs plus modérés de 2022-2026 restent dans la zone « normale ».

## Graphique 8 : Distribution du score d'anomalie, apprentissage et test

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(-modele.decision_function(train[VARIABLES]), bins=60, density=True, alpha=0.6, label="Apprentissage 2016-2021")
ax.hist(test["score"], bins=60, density=True, alpha=0.6, label="Test 2022-...")
ax.axvline(0, color="black", ls="--", label="Seuil de décision")
ax.set_xlabel("Score d'anomalie (sans unité)")
ax.set_ylabel("Densité")
ax.set_title("Distribution du score d'anomalie : apprentissage et test")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / "fig8_scores.png", dpi=150)
plt.show()
print(f"Proportion signalée : apprentissage {100 * detecte_train.mean():.2f} % | test {100 * test['anomalie'].mean():.2f} %")

**Interprétation.** Le modèle signale **0,55 %** des jours en apprentissage et **0 %** au test : toute la distribution des scores du test se situe à gauche du seuil. C'est un **changement de régime** : la « normalité » apprise contient le Covid, dont l'ampleur n'a pas été égalée depuis. Ce n'est pas une erreur de code, mais la limite principale du modèle, qui plaide pour un réentraînement régulier sur une fenêtre récente.